In [1]:
!uv pip install "eigenp-utils[image-analysis, plotting] @ git+https://github.com/eigenP/utils.git"

Using Python 3.12.13 environment at: /usr
Resolved 107 packages in 1.49s
Prepared 1 package in 3.51s
Uninstalled 1 package in 2ms
Installed 1 package in 2ms
 - eigenp-utils==0.0.2 (from git+https://github.com/eigenP/utils.git@2bb3564b010f5e2da9ef89ef07f09607bdd1bf79)
 + eigenp-utils==0.0.2 (from git+https://github.com/eigenP/utils.git@876b4c8767f29eef79d870981f999a2634882e1c)


In [2]:
import numpy as np
from skimage.data import cells3d
from eigenp_utils.tnia_plotting_anywidgets import (
    show_zyx_max_slice_interactive,
    show_zyx_max_slice_interactive_point_annotator
)

# Load the 3D cell data (Z, C, Y, X)
try:
    cells = cells3d()
except:
    from eigenp_utils.io import download_file
    url_to_fetch = "https://gitlab.com/scikit-image/data/-/raw/master/cells3d.tif"
    download_file(url_to_fetch, "./cells3d.tif")
    from skimage.io import imread
    cells = imread("./cells3d.tif")
print(f"Loaded cells3d with shape: {cells.shape}")

# Extract channels
nuclei = cells[:, 1, :, :]
membrane = cells[:, 0, :, :]
PIXEL_SIZES = {'Z' : 0.29 , 'Y' : 0.26, 'X' : 0.26}

Hint: labels_cmap background is transparent by default. To set it to black, run:
labels_cmap._init(); labels_cmap._lut[0] = [0, 0, 0, 1]
Loaded cells3d with shape: (60, 2, 256, 256)


## Interactive 3D Maximum Intensity Projection & Slicing

You can use the sliders below to navigate the 3D volume along any axis. The `Thickness` sliders control the maximum intensity projection slab size, while `Position` navigates the slab center.

In [3]:
# Standard Viewer
viewer = show_zyx_max_slice_interactive(
    [nuclei, membrane],
    colormap=['yellow', 'cyan'],
    vmin=[2_000, 1_000], vmax=[50_000, 25_000],
    slabs_thickness=(nuclei.shape[0] // 4 - 1, 3,None),
    show_crosshair=True,
    sync_on_hover=True,
    pixel_sizes=PIXEL_SIZES,
    # pixel_sizes=(0.29, 0.26, 0.26),
    figsize_scale=0.75
)

viewer

## Point Annotator Widget

Toggle the "ANNOTATION" checkbox to start adding or deleting points. Points added are persistent and sync directly back to the `widget.points` list in Python. (can make polygons / splines / use as inputs for segmentation, etc.)

In [4]:
# Annotator Viewer
annotator = show_zyx_max_slice_interactive_point_annotator(
    [nuclei],
    colormap=['viridis'],
    slabs_thickness=(3,3,None),
    show_crosshair=True,
    pixel_sizes=PIXEL_SIZES,
    figsize_scale=0.5,
    point_size_scale=0.015  # Adjust marker size
)


annotator

In [5]:
annotator.points

[]

In [6]:
from eigenp_utils.image_and_labels_utils import voronoi_otsu_labeling
from eigenp_utils.plotting_utils import labels_cmap
# from matplotlib import colormaps as mpl_colormaps
from skimage.segmentation import find_boundaries


# mpl_colormaps.register(labels_cmap, name='labels_cmap', force=True)


labels = voronoi_otsu_labeling(nuclei, spot_sigma=7, outline_sigma=2)



# mode='inner' ensures the boundary pixels are located inside the original labels
boundaries_mask = find_boundaries(labels, mode='inner')

# Multiply by the original labels to keep the integer labels
boundary_labels = labels * boundaries_mask



viewer_labels = show_zyx_max_slice_interactive(
    [nuclei, boundary_labels],
    colormap=['lime', 'labels_cmap'],
    # vmin=[1_000, 500], vmax=[60_000, 25_000],
    slabs_thickness=(nuclei.shape[0] // 8 - 1, nuclei.shape[1] // 16 - 1,None),
    show_crosshair=True,
    sync_on_hover=True,
    pixel_sizes=PIXEL_SIZES,
    # pixel_sizes=(0.29, 0.26, 0.26),
    figsize_scale=0.75
)

viewer_labels